In [ ]:
%pip install pandas numpy scipy scikit-learn joblib openpyxl skrebate --q

# 2 — Métricas para CEQ e rankings — loop automático por turma (estudo de caso 2)

**Organização por classe**: a base tem uma coluna `Classe` com várias categorias (ex.: A, B, C, D, E). Este notebook:

1. Detecta todas as classes presentes na última coluna da base.
2. Para cada classe, calcula os índices CEQ, os gráficos de controle e todos os métodos de literatura usando essa classe como classe de interesse (1) contra todas as outras juntas (0, lógica *one-vs-rest*), e salva os 43 rankings numa subpasta própria, `Classe_<rótulo>/`.

Cada `Classe_<rótulo>/` contém os 43 `Ranking_*.xlsx`, o `Tempos_Execucao.xlsx` e o `amostra_classe_<rótulo>.xlsx`, e é a entrada dos notebooks seguintes, que processam essas subpastas automaticamente.

1. Detecta sozinho todas as classes presentes na última coluna da base.
2. Para **cada** classe, repete exatamente a mesma lógica de antes — calcula os
   índices CEQ, os gráficos de controle e todos os métodos de literatura usando
   essa classe como "classe escolhida" (1) contra todas as outras juntas (0, a
   mesma lógica *one-vs-rest* de antes) — e salva os 43 rankings dessa classe numa
   subpasta própria, `Classe_<rótulo>/` (ex.: `Classe_A/`, `Classe_B/`...).

Nada na matemática de cada método mudou — é a mesma conta de sempre, só que repetida
automaticamente para cada classe em vez de rodar manualmente, uma vez por classe,
copiando o notebook para pastas diferentes. Os notebooks seguintes (3 a 7) já foram
adaptados para também descobrir e processar essas subpastas sozinhos.

Ao final, cada `Classe_<rótulo>/` fica com os mesmos arquivos que o notebook gerava
antes numa pasta só (os 43 `Ranking_*.xlsx`, `Tempos_Execucao.xlsx` e
`amostra_classe_<rótulo>.xlsx`) — totalmente autocontida, pronta para os notebooks
seguintes.

**Método**: o mesmo do estudo de caso 1 (SECOM): LSL/USL = percentis 1 e 99 de cada atributo na classe de referência; Cp/Cpk com σ de curto prazo (amplitude móvel) e Pp/Ppk com σ de longo prazo; índice indefinido (atributo constante) = NaN, que vai para o fim do ranking. Entrada: a aba `Amostra` de `amostra.xlsx` (notebook 1).


### 1. Carregar dados e preparar o que é comum a TODAS as classes

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, spearmanr, kendalltau, ttest_ind, ks_2samp, entropy
from scipy.linalg import svd, eigh
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, mutual_info_classif, f_classif
from sklearn.metrics import mutual_info_score
from skrebate import ReliefF

# 1. Carregar arquivo
df = pd.read_excel("amostra.xlsx")
df.columns = df.columns.astype(str).str.strip()

coluna_alvo = df.columns[-1]
X = df.iloc[:, :-1].reset_index(drop=True)
y = df.iloc[:, -1].reset_index(drop=True)

# 0) Sensores SEM NENHUMA leitura válida em TODA a amostra (todas as classes) --
# nenhuma mediana (nem global, nem por classe) resolve isso, então removemos essas
# colunas da análise como um todo, de forma explícita e registrada (em vez de deixar
# NaN residual quebrar SVD Entropy/Laplacian/SPEC mais à frente).
colunas_numericas_df = df.select_dtypes(include=[np.number]).columns
colunas_totalmente_vazias = colunas_numericas_df[df[colunas_numericas_df].isna().all()]
if len(colunas_totalmente_vazias) > 0:
    df = df.drop(columns=colunas_totalmente_vazias)
    X = X.drop(columns=[c for c in colunas_totalmente_vazias if c in X.columns])
    print(f"{len(colunas_totalmente_vazias)} sensor(es) sem NENHUMA leitura válida em toda a "
          f"amostra foram removidos da análise: {list(colunas_totalmente_vazias)}")

# Atributos numéricos (iguais para todas as classes)
X_quant = X.select_dtypes(include=[np.number]).copy()
n_ausentes_x = int(X_quant.isna().sum().sum())
if n_ausentes_x > 0:
    X_quant = X_quant.fillna(X_quant.median())
print(f"Imputação defensiva (X_quant): {n_ausentes_x} valor(es) ausente(s) preenchido(s).")

# Normalização dos dados (igual para todas as classes)
X_norm = pd.DataFrame(MinMaxScaler().fit_transform(X_quant), columns=X_quant.columns)

# Base completa (todas as classes), sem o rótulo, com ausentes preenchidos --
# usada pelos métodos supervisionados de literatura (Pearson, ANOVA, Chi² etc.)
X_full = df.drop(columns=[coluna_alvo]).select_dtypes(include=[np.number]).copy()
X_full = X_full.fillna(X_full.median())
X_scaled_full = MinMaxScaler().fit_transform(X_full)  # reusado por Pearson/Spearman/Kendall/Chi2/ReliefF

# Versão "_BaseCompleta" das descritivas (todas as classes) -- não depende da classe
# escolhida, calculada 1x e gravada em cada subpasta de classe.
df_numerico_completo = X_full.copy()

# Classes disponíveis -- detectadas automaticamente (antes era 1 input() por execução)
classes_disponiveis = sorted(df.iloc[:, -1].dropna().unique().tolist())
print(f"Classes encontradas na coluna '{coluna_alvo}': {classes_disponiveis}")

nome_base_cenario = input(
    "Nome base do cenário/base (ex.: Registro Equipamento Jan). "
    "Deixe em branco para usar um nome genérico: "
) or "Base"

MAX_AMOSTRAS_NOTEBOOK1 = 500  # mesmo teto usado no notebook 1 (amostragem adaptativa)


### Funções auxiliares (comuns a todos os métodos)

`montar_ranking` padroniza toda métrica em 3 colunas — `Feature`, `Score`, `Ranking`
— já ordenadas do melhor (Ranking 1) para o pior atributo. `salvar_ranking` grava o
Excel **dentro da subpasta da classe sendo processada** (`out_dir`) e acumula o tempo
de execução numa lista em memória — em vez de reabrir e reescrever
`Tempos_Execucao.xlsx` a cada chamada (43 vezes por classe), a lista só é despejada em
Excel uma vez, ao final de cada classe (seção "Execução", no fim do notebook).

In [ ]:
def montar_ranking(features, scores, ascending=False):
    """Padroniza a saída em 3 colunas: Feature, Score, Ranking (1 = melhor atributo)."""
    ranking_df = pd.DataFrame({"Feature": features, "Score": scores})
    ranking_df = ranking_df.sort_values(by="Score", ascending=ascending, na_position="last").reset_index(drop=True)
    ranking_df["Ranking"] = ranking_df.index + 1
    return ranking_df


def salvar_ranking(ranking_df, nome_arquivo, tempo_execucao, out_dir, lista_tempos, mensagem=None):
    """Salva o ranking (já no formato Feature/Score/Ranking) dentro de `out_dir`
    (a subpasta Classe_<rótulo>/ sendo processada) e registra o tempo em memória."""
    caminho = os.path.join(out_dir, nome_arquivo)
    ranking_df.to_excel(caminho, index=False)
    print(mensagem or f"Ranking salvo em '{caminho}'")
    lista_tempos.append((nome_arquivo, tempo_execucao))


### Bloco 1 — CEQ (Cp/Cpk/Pp/Ppk) e Gráficos de Controle (p/np/c/u)

Calculado só sobre os exemplos da **classe sendo processada** (análogo ao que antes
era "a classe escolhida pelo usuário"). Cp/Cpk usam sigma de **curto prazo** (amplitude
móvel); Pp/Ppk usam sigma de **longo prazo** (desvio-padrão amostral) — a diferença
entre eles é o que permite avaliar estabilidade do processo dentro de cada classe.

In [ ]:
# Limites de especificação de Cp/Cpk/Pp/Ppk (mesma regra do estudo de caso 1)
# Os dados dos caminhões não têm limites de especificação de engenharia. Na ausência
# deles, LSL e USL são estimados pelos PERCENTIS 1 e 99 de cada atributo na classe
# (turma) de referência: a faixa que contém 98% das leituras, ignorando 1% dos valores
# mais extremos de cada lado. Com mínimo/máximo, uma única leitura atípica mudava o
# limite, o ranking não era reprodutível e Pp = (máx-mín)/6σ ficava idêntico a
# AmplitudeSigma/6 (o grupo de controle).
# O notebook 1 usa os MESMOS percentis na regra de parada -- mude os dois juntos.
PERCENTIL_LSL = 1
PERCENTIL_USL = 99

def sigma_curto_prazo_mr(dados):
    """Estima o sigma de CURTO PRAZO (variação inerente) via amplitude móvel (moving
    range) entre observações consecutivas -- técnica padrão de um gráfico I-MR quando
    não há subgrupos racionais definidos (Montgomery, 2020). PREMISSA: assume que a
    ordem das linhas reflete a ordem real de coleta/processo."""
    valores = dados.values
    if len(valores) < 2:
        return np.nan
    mr = np.abs(np.diff(valores))
    return np.mean(mr) / 1.128  # d2 para subgrupo de tamanho 2


def calcular_ceq(df_classe):
    """Cp/Cpk usam sigma de CURTO prazo (moving range) -- 'capacidade potencial'.
    Pp/Ppk usam sigma de LONGO prazo (desvio-padrão amostral geral) -- 'desempenho
    real'. Diferente de antes, as quatro fórmulas NÃO são mais idênticas entre si:
    a diferença Cp vs. Pp (e Cpk vs. Ppk) é justamente o que permite avaliar
    estabilidade -- Cp ≈ Pp indica processo estável; Cp >> Pp indica variação extra
    de longo prazo (deriva, instabilidade) não capturada pela variação ponto-a-ponto.
    """
    resultados = []
    for col in df_classe.columns:
        dados = df_classe[col].dropna()
        LSL, USL = np.percentile(dados, [PERCENTIL_LSL, PERCENTIL_USL]) if len(dados) else (np.nan, np.nan)
        media = dados.mean()

        std_longo = np.std(dados, ddof=0)          # usado por Pp/Ppk
        std_curto = sigma_curto_prazo_mr(dados)     # usado por Cp/Cpk

        invalido = (
            len(dados) < 2 or std_longo == 0
            or std_curto == 0 or pd.isna(std_curto)
        )
        if invalido:
            # Índice indefinido (sensor constante): NaN -> fim do ranking. Antes era 0,
            # o que o deixava ACIMA de sensores com Cpk/Ppk negativo (média fora de [P1, P99]).
            valores = {m: np.nan for m in ["Cp", "Cpk", "Pp", "Ppk"]}
        else:
            Cp = (USL - LSL) / (6 * std_curto)
            Cpu = (USL - media) / (3 * std_curto)
            Cpl = (media - LSL) / (3 * std_curto)
            Cpk = min(Cpu, Cpl)
            Pp = (USL - LSL) / (6 * std_longo)
            Ppu = (USL - media) / (3 * std_longo)
            Ppl = (media - LSL) / (3 * std_longo)
            Ppk = min(Ppu, Ppl)
            valores = {"Cp": Cp, "Cpk": Cpk, "Pp": Pp, "Ppk": Ppk}

        resultados.append({"Feature": col, **valores})
    return pd.DataFrame(resultados)


def calcular_graficos_controle(df_classe):
    n = len(df_classe)
    colunas = df_classe.columns

    # Matriz binária de "defeito": 1 onde o valor sai de ±1σ da média da
    # PRÓPRIA coluna -- calculada de uma vez para TODOS os atributos, pois
    # c/u (abaixo) precisam saber quantos atributos estão fora de controle
    # simultaneamente em cada amostra, não só o atributo atual.
    matriz_defeito = pd.DataFrame(0, index=df_classe.index, columns=colunas, dtype=int)
    for col in colunas:
        dados_col = df_classe[col]
        dados_validos = dados_col.dropna()
        media = dados_validos.mean()
        std = np.std(dados_validos, ddof=0)  # consistente com o ddof=0 de calcular_ceq(), acima
        if len(dados_validos) >= 2 and std > 0:
            fora_de_controle = (dados_col < media - std) | (dados_col > media + std)
            matriz_defeito[col] = fora_de_controle.fillna(False).astype(int)

    # D_i: nº TOTAL de atributos fora de controle, simultaneamente, na amostra i
    # (conceito clássico de "defeitos por unidade" -- uma unidade pode ter
    # vários defeitos ao mesmo tempo, em características/atributos diferentes).
    defeitos_simultaneos_por_amostra = matriz_defeito.sum(axis=1)

    resultados = []
    for col in colunas:
        dados = df_classe[col].dropna()
        std = np.std(dados, ddof=0)

        if len(dados) < 2 or std == 0:
            valores = {m: np.nan for m in ["p", "np", "c", "u"]}   # indefinido -> fim do ranking
        else:
            amostras_defeituosas_nesta_coluna = matriz_defeito[col].astype(bool)
            n_defeituosas = int(amostras_defeituosas_nesta_coluna.sum())

            p = n_defeituosas / n
            np_val = n_defeituosas
            c = int(defeitos_simultaneos_por_amostra[amostras_defeituosas_nesta_coluna].sum())
            u = c / n

            valores = {"p": p, "np": np_val, "c": c, "u": u}

        resultados.append({"Feature": col, **valores})
    return pd.DataFrame(resultados)


def gerar_bloco1_ceq_controle(base_classe1_orig, base_classe1_norm, out_dir, lista_tempos):
    for normalizado, base_classe1 in zip([False, True], [base_classe1_orig, base_classe1_norm]):
        df_ceq = calcular_ceq(base_classe1)
        for metrica in ["Cp", "Cpk", "Pp", "Ppk"]:
            t0 = time.time()
            df_rank = montar_ranking(df_ceq["Feature"], df_ceq[metrica], ascending=False)
            nome = f"Ranking_{metrica}_{'Normalizado' if normalizado else 'Original'}.xlsx"
            salvar_ranking(df_rank, nome, time.time() - t0, out_dir, lista_tempos)
        df_gc = calcular_graficos_controle(base_classe1)
        for metrica in ["p", "np", "c", "u"]:
            t0 = time.time()
            df_rank = montar_ranking(df_gc["Feature"], df_gc[metrica], ascending=False)
            nome = f"Ranking_{metrica}_{'Normalizado' if normalizado else 'Original'}.xlsx"
            salvar_ranking(df_rank, nome, time.time() - t0, out_dir, lista_tempos)


### Estatísticas descritivas: grupo de controle do CEQ

Duas versões de cada estatística (Amplitude/σ, IQR/σ, Assimetria, Curtose):

- **`Ranking_<Nome>.xlsx`** — calculada só sobre a classe sendo processada, mesma
  população do CEQ (controle pareado, usado no notebook 3).
- **`Ranking_<Nome>_BaseCompleta.xlsx`** — sobre a base inteira (todas as classes),
  igual em TODAS as subpastas de classe (não muda com a classe escolhida); serve de
  referência, mas não é pareada com o CEQ.

In [ ]:
def amplitude_sigma(coluna):
    x = coluna.dropna().values
    if len(x) < 2:
        return np.nan
    std = np.std(x, ddof=0)
    return (np.max(x) - np.min(x)) / std if std > 0 else np.nan


def iqr_sigma(coluna):
    x = coluna.dropna().values
    if len(x) < 2:
        return np.nan
    std = np.std(x, ddof=0)
    q1, q3 = np.percentile(x, [25, 75])
    return (q3 - q1) / std if std > 0 else np.nan


def assimetria_abs(coluna):
    x = coluna.dropna().values
    if len(x) < 3:
        return np.nan
    return abs(skew(x, bias=True))


def curtose_abs(coluna):
    x = coluna.dropna().values
    if len(x) < 3:
        return np.nan
    return abs(kurtosis(x, fisher=True, bias=True))


metricas_comparacao = {
    "AmplitudeSigma": amplitude_sigma, "IQRSigma": iqr_sigma,
    "Assimetria": assimetria_abs, "Curtose": curtose_abs,
}


def gerar_descritivas(df_numerico_comp, df_numerico_completo, out_dir, lista_tempos):
    versoes = [("", df_numerico_comp), ("_BaseCompleta", df_numerico_completo)]
    for nome_metrica, func in metricas_comparacao.items():
        for sufixo, base_calc in versoes:
            t0 = time.time()
            scores = [func(base_calc[col]) for col in base_calc.columns]
            ranking_df = montar_ranking(base_calc.columns, scores, ascending=False)
            nome_arq = f"Ranking_{nome_metrica}{sufixo}.xlsx"
            salvar_ranking(ranking_df, nome_arq, time.time() - t0, out_dir, lista_tempos)


### ANOVA F-value manual\n\n*Maior F-score = melhor atributo (mais separação entre a classe escolhida e o resto).*

In [ ]:
def anova_f_score(X, y):
    scores = []
    classes = np.unique(y)
    overall_means = np.mean(X, axis=0)
    for i in range(X.shape[1]):
        between = within = 0.0
        for c in classes:
            X_c = X[y == c, i]
            mean_c = np.mean(X_c)
            between += len(X_c) * (mean_c - overall_means[i]) ** 2
            within += np.sum((X_c - mean_c) ** 2)
        scores.append(between / within if within != 0 else 0)
    return scores


def gerar_anova(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    f_scores = anova_f_score(X_full.values, y_bin.values)
    ranking_df = montar_ranking(X_full.columns, f_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_ANOVA.xlsx", time.time() - t0, out_dir, lista_tempos)


### SVD Entropy\n\n*Métrica intrínseca (não usa y) -- maior entropia = melhor atributo. Calculada sobre a classe sendo processada.*

In [ ]:
def svd_entropy(vetor):
    vetor = np.array(vetor)
    if vetor.ndim == 1:
        vetor = vetor.reshape(1, -1)
    U, S, Vh = svd(vetor, full_matrices=False)
    P = S / np.sum(S)
    return entropy(P)


def gerar_svd_entropy(df_classe, out_dir, lista_tempos):
    t0 = time.time()
    df_numerico = df_classe.select_dtypes(include=[np.number])
    entropias = [svd_entropy(df_numerico[col].values) for col in df_numerico.columns]
    ranking_df = montar_ranking(df_numerico.columns, entropias, ascending=False)
    salvar_ranking(ranking_df, "Ranking_SVDEntropy.xlsx", time.time() - t0, out_dir, lista_tempos)


### Laplacian Score\n\n*Métrica intrínseca -- menor score = melhor atributo (mais discriminativo na vizinhança local).*

In [ ]:
def _grafo_vizinhanca(X, k=5):
    n_samples = X.shape[0]
    dist_matrix = squareform(pdist(X, metric="euclidean"))
    W = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        idx = np.argsort(dist_matrix[i])[1:k + 1]
        W[i, idx] = 1
        W[idx, i] = 1
    D = np.diag(W.sum(axis=1))
    L = D - W
    return D, L


def laplacian_score(X, k=5):
    D, L = _grafo_vizinhanca(X, k)
    scores = []
    for i in range(X.shape[1]):
        f = X[:, i]
        f = f - np.mean(f)
        numerator = f.T @ L @ f
        denominator = f.T @ D @ f + 1e-12
        scores.append(numerator / denominator)
    return scores


def gerar_laplacian(df_classe, out_dir, lista_tempos):
    t0 = time.time()
    Xc = df_classe.select_dtypes(include=[np.number]).values
    atributos = df_classe.select_dtypes(include=[np.number]).columns
    scores = laplacian_score(Xc)
    ranking_df = montar_ranking(atributos, scores, ascending=True)
    salvar_ranking(ranking_df, "Ranking_LaplacianScore.xlsx", time.time() - t0, out_dir, lista_tempos)


### SPEC Score\n\n*Métrica intrínseca -- maior score = melhor atributo.*

In [ ]:
def spec_score(X, k=5, n_eig=5):
    D, L = _grafo_vizinhanca(X, k)
    eigvals, eigvecs = eigh(L)
    eigvecs_selected = eigvecs[:, 1:n_eig + 1]
    scores = []
    for i in range(X.shape[1]):
        f = X[:, i]
        f = (f - np.mean(f)) / (np.std(f) + 1e-12)
        proj = eigvecs_selected.T @ f
        scores.append(np.sum(proj ** 2))
    return scores


def gerar_spec(df_classe, out_dir, lista_tempos):
    t0 = time.time()
    Xc = df_classe.select_dtypes(include=[np.number]).values
    atributos = df_classe.select_dtypes(include=[np.number]).columns
    scores = spec_score(Xc)
    ranking_df = montar_ranking(atributos, scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_SPEC.xlsx", time.time() - t0, out_dir, lista_tempos)


## ReliefF\n\n*Maior score = melhor atributo.*

In [ ]:
def gerar_relieff(X_full, X_scaled_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    relieff = ReliefF(n_neighbors=10, n_features_to_select=X_full.shape[1])
    relieff.fit(X_scaled_full, y_bin.values)
    ranking_df = montar_ranking(X_full.columns, relieff.feature_importances_, ascending=False)
    salvar_ranking(ranking_df, "Ranking_ReliefF.xlsx", time.time() - t0, out_dir, lista_tempos)


### Algoritmos univariados (literatura)\n\n### Correlação de Pearson\n\n*Maior |r| = melhor atributo.*

In [ ]:
def gerar_pearson(X_full, X_scaled_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    correlacoes = [abs(np.corrcoef(X_scaled_full[:, i], y_bin.values)[0, 1]) for i in range(X_full.shape[1])]
    ranking_df = montar_ranking(X_full.columns, correlacoes, ascending=False)
    salvar_ranking(ranking_df, "Ranking_Pearson.xlsx", time.time() - t0, out_dir, lista_tempos)


### Correlação de Spearman ρ\n\n*Maior |ρ| = melhor atributo.*

In [ ]:
def gerar_spearman(X_full, X_scaled_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    correlacoes = []
    for i in range(X_full.shape[1]):
        rho, _ = spearmanr(X_scaled_full[:, i], y_bin.values)
        correlacoes.append(abs(rho))
    ranking_df = montar_ranking(X_full.columns, correlacoes, ascending=False)
    salvar_ranking(ranking_df, "Ranking_Spearman.xlsx", time.time() - t0, out_dir, lista_tempos)


### Correlação de Kendall τ\n\n*Maior |τ| = melhor atributo.*

In [ ]:
def gerar_kendall(X_full, X_scaled_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    correlacoes = []
    for i in range(X_full.shape[1]):
        tau, _ = kendalltau(X_scaled_full[:, i], y_bin.values)
        correlacoes.append(abs(tau))
    ranking_df = montar_ranking(X_full.columns, correlacoes, ascending=False)
    salvar_ranking(ranking_df, "Ranking_Kendall.xlsx", time.time() - t0, out_dir, lista_tempos)


### Qui-quadrado (χ²)\n\n*Maior estatística = melhor atributo.*

In [ ]:
def gerar_chi2(X_full, X_scaled_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    chi2_scores, p_values = chi2(X_scaled_full, y_bin.values)
    ranking_df = montar_ranking(X_full.columns, chi2_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_Chi2.xlsx", time.time() - t0, out_dir, lista_tempos)


### mutual_info_classif\n\n*Maior informação mútua = melhor atributo.*

In [ ]:
def gerar_infogain(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    info_gain_scores = mutual_info_classif(X_full, y_bin.values, discrete_features=False)
    ranking_df = montar_ranking(X_full.columns, info_gain_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_InformationGain.xlsx", time.time() - t0, out_dir, lista_tempos)


### Gain Ratio\n\n*Maior razão de ganho = melhor atributo.*

In [ ]:
def _intrinsic_value(column):
    bins = pd.qcut(column, q=10, duplicates="drop")
    probs = bins.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs + 1e-9))


def gerar_gainratio(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    info_gain_scores = mutual_info_classif(X_full, y_bin.values, discrete_features=False)
    intrinsic_values = [_intrinsic_value(X_full[col]) for col in X_full.columns]
    gain_ratios = [ig / iv if iv != 0 else 0 for ig, iv in zip(info_gain_scores, intrinsic_values)]
    ranking_df = montar_ranking(X_full.columns, gain_ratios, ascending=False)
    salvar_ranking(ranking_df, "Ranking_GainRatio.xlsx", time.time() - t0, out_dir, lista_tempos)


### F-Score (ANOVA via sklearn)\n\n*Maior F-score = melhor atributo.*

In [ ]:
def gerar_fscore_anova(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    f_scores, p_values = f_classif(X_full, y_bin.values)
    ranking_df = montar_ranking(X_full.columns, f_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_FScore_ANOVA.xlsx", time.time() - t0, out_dir, lista_tempos)


### T-Score (teste t)\n\n*Maior |t| = melhor atributo.*

In [ ]:
def gerar_tscore(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    classes = np.unique(y_bin.values)  # sempre [0, 1] com a base binária (classe vs. resto)
    t_scores = []
    for col in X_full.columns:
        g1 = X_full[y_bin.values == classes[0]][col]
        g2 = X_full[y_bin.values == classes[1]][col]
        t_stat, p_val = ttest_ind(g1, g2, equal_var=False)
        t_scores.append(abs(t_stat))
    ranking_df = montar_ranking(X_full.columns, t_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_TScore.xlsx", time.time() - t0, out_dir, lista_tempos)


### Fisher Score\n\n*Maior score = melhor atributo.*

In [ ]:
def gerar_fisherscore(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    classes = np.unique(y_bin.values)
    fisher_scores = []
    for col in X_full.columns:
        overall_mean = X_full[col].mean()
        numerator = denominator = 0
        for c in classes:
            class_data = X_full[y_bin.values == c][col]
            class_mean = class_data.mean()
            class_var = class_data.var(ddof=1)
            numerator += len(class_data) * (class_mean - overall_mean) ** 2
            denominator += len(class_data) * class_var
        fisher_scores.append(numerator / (denominator + 1e-9))
    ranking_df = montar_ranking(X_full.columns, fisher_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_FisherScore.xlsx", time.time() - t0, out_dir, lista_tempos)


### Gini Index\n\n*Menor índice = melhor atributo (mais pureza nas classes).*

In [ ]:
def gerar_giniindex(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    classes = np.unique(y_bin.values)

    def gini_index(attribute, target):
        bins = pd.qcut(attribute, q=10, duplicates="drop")
        gini = 0
        for b in bins.unique():
            subset = target[bins == b]
            if len(subset) == 0:
                continue
            probs = np.bincount(subset, minlength=len(classes)) / len(subset)
            gini_bin = 1 - np.sum(probs ** 2)
            gini += (len(subset) / len(target)) * gini_bin
        return gini

    gini_scores = [gini_index(X_full[col], y_bin.values) for col in X_full.columns]
    ranking_df = montar_ranking(X_full.columns, gini_scores, ascending=True)
    salvar_ranking(ranking_df, "Ranking_GiniIndex.xlsx", time.time() - t0, out_dir, lista_tempos)


### Symmetrical Uncertainty\n\n*Maior valor = melhor atributo.*

In [ ]:
def _entropy_discrete(values):
    probs = np.bincount(values) / len(values)
    return entropy(probs, base=2)


def _entropy_continuous(column):
    bins = pd.qcut(column, q=10, duplicates="drop")
    probs = bins.value_counts(normalize=True)
    return entropy(probs, base=2)


def gerar_symmetricaluncertainty(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    info_gain = mutual_info_classif(X_full, y_bin.values, discrete_features=False)
    H_y = _entropy_discrete(y_bin.values)
    H_x = [_entropy_continuous(X_full[col]) for col in X_full.columns]
    su_scores = [2 * ig / (hx + H_y + 1e-9) for ig, hx in zip(info_gain, H_x)]
    ranking_df = montar_ranking(X_full.columns, su_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_SymmetricalUncertainty.xlsx", time.time() - t0, out_dir, lista_tempos)


### MIC (Maximal Information Coefficient) — aproximado\n\n*Maior valor = melhor atributo.*

In [ ]:
def _approximate_mic(x, y, bins=10):
    try:
        x_binned = pd.qcut(x, q=bins, duplicates="drop", labels=False).fillna(-1)
    except ValueError:
        x_binned = pd.cut(x, bins=bins, labels=False).fillna(-1)
    if len(np.unique(y)) > bins:
        try:
            y_binned = pd.qcut(y, q=bins, duplicates="drop", labels=False).fillna(-1)
        except ValueError:
            y_binned = pd.cut(y, bins=bins, labels=False).fillna(-1)
    else:
        y_binned = y
    mi = mutual_info_score(x_binned, y_binned)
    norm_factor = np.log2(min(len(np.unique(x_binned)), len(np.unique(y_binned))) + 1e-9)
    return mi / norm_factor if norm_factor > 0 else 0


def gerar_approxmic(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    mic_scores = [_approximate_mic(X_full[col], y_bin.values) for col in X_full.columns]
    ranking_df = montar_ranking(X_full.columns, mic_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_ApproxMIC.xlsx", time.time() - t0, out_dir, lista_tempos)


### Kolmogorov–Smirnov\n\n*Maior estatística KS = melhor atributo.*

In [ ]:
def gerar_ks(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    classes = np.unique(y_bin.values)
    ks_scores = []
    for col in X_full.columns:
        g1 = X_full[y_bin.values == classes[0]][col]
        g2 = X_full[y_bin.values == classes[1]][col]
        ks_stat, p_val = ks_2samp(g1, g2)
        ks_scores.append(ks_stat)
    ranking_df = montar_ranking(X_full.columns, ks_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_KS.xlsx", time.time() - t0, out_dir, lista_tempos)


### Hellinger Distance\n\n*Maior distância = melhor atributo.*

In [ ]:
def _hellinger(p, q):
    return np.sqrt(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2)) / np.sqrt(2)


def gerar_hellinger(X_full, y_bin, out_dir, lista_tempos):
    t0 = time.time()
    classes = np.unique(y_bin.values)

    def compute_hellinger(feature, yv, bins=10):
        g1 = feature[yv == classes[0]]
        g2 = feature[yv == classes[1]]
        hist1, _ = np.histogram(g1, bins=bins, range=(feature.min(), feature.max()), density=True)
        hist2, _ = np.histogram(g2, bins=bins, range=(feature.min(), feature.max()), density=True)
        p = hist1 / np.sum(hist1)
        q = hist2 / np.sum(hist2)
        return _hellinger(p, q)

    hellinger_scores = [compute_hellinger(X_full[col], y_bin.values) for col in X_full.columns]
    ranking_df = montar_ranking(X_full.columns, hellinger_scores, ascending=False)
    salvar_ranking(ranking_df, "Ranking_HellingerDistance.xlsx", time.time() - t0, out_dir, lista_tempos)


### Execução — loop automático por todas as classes

Para cada classe detectada: monta `df_classe` (imputação defensiva igual à versão
anterior — usa mediana global quando um sensor não tem NENHUMA leitura válida dentro
da classe) e `y_bin` (classe escolhida vs. todas as outras), roda os 43 métodos acima
e grava tudo em `Classe_<rótulo>/`. Ao final, salva também uma aba
`Info_Amostral_CODE10` em `amostra.xlsx` com 1 linha por classe (tamanho amostral de
cada uma).

In [ ]:
rotulo_col = coluna_alvo
info_amostral_registros = []

for classe_desejada in classes_disponiveis:
    t_classe_inicio = time.time()
    nome_cenario = f"{nome_base_cenario} - Classe {classe_desejada}"
    out_dir = f"Classe_{classe_desejada}"
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{'=' * 70}\n Processando CLASSE {classe_desejada}  ->  pasta '{out_dir}/'\n{'=' * 70}")

    lista_tempos = []

    # Imputação defensiva por mediana em df_classe (mediana da própria classe; se
    # algum sensor não tiver NENHUMA leitura válida dentro dela, usa a mediana global)
    df_classe = df[df.iloc[:, -1].astype(str) == str(classe_desejada)].reset_index(drop=True)
    colunas_numericas_classe = df_classe.select_dtypes(include=[np.number]).columns
    mediana_classe = df_classe[colunas_numericas_classe].median()
    mediana_global = df[colunas_numericas_classe].median()
    n_ausentes_classe = int(df_classe[colunas_numericas_classe].isna().sum().sum())
    if n_ausentes_classe > 0:
        mediana_para_preencher = mediana_classe.fillna(mediana_global)
        df_classe[colunas_numericas_classe] = df_classe[colunas_numericas_classe].fillna(mediana_para_preencher)

    arquivo_classe = os.path.join(out_dir, f"amostra_classe_{classe_desejada}.xlsx")
    df_classe.to_excel(arquivo_classe, index=False)

    # Base BINÁRIA: classe escolhida (1) vs. todas as outras (0)
    y_bin = (df.iloc[:, -1].astype(str) == str(classe_desejada)).astype(int).reset_index(drop=True)
    print(f"Classe {classe_desejada}: {int((y_bin == 1).sum())} amostras na classe, "
          f"{int((y_bin == 0).sum())} nas demais | salvo em '{arquivo_classe}'")

    info_amostral_registros.append({
        "Base/Cenário": nome_cenario, "Classe": classe_desejada,
        "n_total_amostra": len(df), "n_classe": int((y_bin == 1).sum()),
        "atingiu_teto_500": len(df) >= MAX_AMOSTRAS_NOTEBOOK1,
        "reamostragem_com_reposicao": False,  # notebook 1 não usa replace=True em nenhum ponto
    })

    # --- Bloco 1: CEQ + gráficos de controle (orig + normalizado) ---
    base_classe1_orig = X_quant[y.astype(str) == str(classe_desejada)].reset_index(drop=True)
    base_classe1_norm = X_norm[y.astype(str) == str(classe_desejada)].reset_index(drop=True)
    gerar_bloco1_ceq_controle(base_classe1_orig, base_classe1_norm, out_dir, lista_tempos)

    # --- Descritivas (controle pareado + BaseCompleta) ---
    df_numerico_comp = df_classe.drop(columns=[rotulo_col], errors="ignore").select_dtypes(include=[np.number])
    gerar_descritivas(df_numerico_comp, df_numerico_completo, out_dir, lista_tempos)

    # --- Literatura ---
    gerar_anova(X_full, y_bin, out_dir, lista_tempos)
    gerar_svd_entropy(df_classe, out_dir, lista_tempos)
    gerar_laplacian(df_classe, out_dir, lista_tempos)
    gerar_spec(df_classe, out_dir, lista_tempos)
    gerar_relieff(X_full, X_scaled_full, y_bin, out_dir, lista_tempos)
    gerar_pearson(X_full, X_scaled_full, y_bin, out_dir, lista_tempos)
    gerar_spearman(X_full, X_scaled_full, y_bin, out_dir, lista_tempos)
    gerar_kendall(X_full, X_scaled_full, y_bin, out_dir, lista_tempos)
    gerar_chi2(X_full, X_scaled_full, y_bin, out_dir, lista_tempos)
    gerar_infogain(X_full, y_bin, out_dir, lista_tempos)
    gerar_gainratio(X_full, y_bin, out_dir, lista_tempos)
    gerar_fscore_anova(X_full, y_bin, out_dir, lista_tempos)
    gerar_tscore(X_full, y_bin, out_dir, lista_tempos)
    gerar_fisherscore(X_full, y_bin, out_dir, lista_tempos)
    gerar_giniindex(X_full, y_bin, out_dir, lista_tempos)
    gerar_symmetricaluncertainty(X_full, y_bin, out_dir, lista_tempos)
    gerar_approxmic(X_full, y_bin, out_dir, lista_tempos)
    gerar_ks(X_full, y_bin, out_dir, lista_tempos)
    gerar_hellinger(X_full, y_bin, out_dir, lista_tempos)

    pd.DataFrame(lista_tempos, columns=["Arquivo", "Tempo (s)"]).to_excel(
        os.path.join(out_dir, "Tempos_Execucao.xlsx"), index=False)
    print(f"Classe {classe_desejada}: {len(lista_tempos)} ranking(s) salvos em '{out_dir}/' "
          f"({time.time() - t_classe_inicio:.1f}s)")

# Info amostral consolidada (1 linha por classe), gravada como aba dentro de amostra.xlsx
df_info = pd.DataFrame(info_amostral_registros)
with pd.ExcelWriter("amostra.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_info.to_excel(writer, sheet_name="Info_Amostral_CODE10", index=False)
print(f"\nInfo amostral (1 linha por classe) salva na aba 'Info_Amostral_CODE10' de amostra.xlsx")

print(f"\nCONCLUÍDO -- rankings gerados para todas as classes: {classes_disponiveis}")
